# Lesson 1: Particle in a 1D Box
## From the Schrödinger Equation to a Quantitative Solver

**Goal:** Build a complete pipeline — concept → math → numerical implementation → validated results — for the simplest quantum system. Establish habits and tools that scale to harder problems.

**What you will learn:**
- The physical content of the Schrödinger equation (time-dependent and time-independent)
- Why quantum time evolution is fundamentally different from classical diffusion
- How to translate a continuous differential equation into a discrete matrix problem
- How to choose units and grid parameters for physically meaningful results
- How to validate numerical results against analytical solutions

**Prerequisites:** Basic calculus, linear algebra (matrix multiplication, eigenvalues), some Python/NumPy experience.

---
## Part A: The Schrödinger Equation — What It Says and Why

### A.1 The Time-Dependent Schrödinger Equation

The fundamental equation of quantum mechanics is:

$$i\hbar \frac{\partial \psi(x,t)}{\partial t} = \hat{H}\, \psi(x,t)$$

Read this as an *equation of motion*: **if we know the state of a system (encoded in the wavefunction $\psi$), the Hamiltonian tells us how that state changes in time.**

Each piece has a physical meaning:

| Symbol | What it is | Physical meaning |
|--------|-----------|-----------------|
| $\psi(x,t)$ | Wavefunction | Complete description of the system's state |
| $\hat{H}$ | Hamiltonian operator | Total energy operator (kinetic + potential) |
| $i$ | Imaginary unit | Ensures oscillation, not decay (see §A.2) |
| $\hbar$ | Reduced Planck constant | Sets the scale where quantum effects matter |

For a single particle in one dimension:

$$\hat{H} = -\frac{\hbar^2}{2m}\frac{\partial^2}{\partial x^2} + V(x)$$

The first term is kinetic energy (related to curvature of the wavefunction). The second is potential energy from the environment the particle lives in.

### A.2 Why the $i$? Oscillation vs. Decay

The $i$ in the Schrödinger equation is not a mathematical convenience — it determines the *character* of quantum time evolution. Compare:

| Equation | Form | Behavior |
|----------|------|----------|
| Diffusion (heat) | $\frac{\partial u}{\partial t} = D \nabla^2 u$ | Decay, smoothing, irreversible |
| Schrödinger | $\frac{\partial \psi}{\partial t} = \frac{-i}{\hbar} \hat{H}\psi$ | Oscillation, preservation, reversible |

**Diffusion:** A spike in temperature spreads out and flattens. Information is lost. The system approaches equilibrium.

**Quantum evolution:** The state *rotates* in the complex plane. Probability is conserved — nothing leaks away. The evolution is reversible.

Mathematically: a real coefficient in the exponential gives decay ($e^{-\alpha t} \to 0$). An imaginary coefficient gives oscillation ($e^{-i\alpha t}$ stays on the unit circle forever). The $i$ is what makes quantum mechanics quantum rather than thermodynamics.

### A.3 The Exponential from Infinitesimal Steps

Where does the exponential time-evolution operator come from? It follows from the logic of small steps.

If the Schrödinger equation says the rate of change of $\psi$ is proportional to $\hat{H}\psi$, then over a tiny time step $\Delta t$:

$$\psi(t + \Delta t) \approx \psi(t) - \frac{i\hat{H}\Delta t}{\hbar}\, \psi(t) = \left(1 - \frac{i\hat{H}\Delta t}{\hbar}\right)\psi(t)$$

After $n$ such steps covering a total time $t = n\,\Delta t$:

$$\psi(t) \approx \left(1 - \frac{i\hat{H}t}{n\hbar}\right)^n \psi(0)$$

In the limit $n \to \infty$, this is the definition of the matrix exponential:

$$\psi(t) = e^{-i\hat{H}t/\hbar}\, \psi(0)$$

**Key insight:** The exponential is not a mysterious mathematical object — it is what you get when a linear "update rule" is applied infinitely many times over infinitesimally small steps. This is the same logic behind compound interest, radioactive decay, and population growth. The new ingredient in quantum mechanics is the $i$, which turns growth/decay into rotation.

### A.4 The Numerical Trap: Finite Steps Break Unitarity

On a computer, we cannot take infinitely many infinitesimal steps. The simplest approach — Euler's method — uses finite $\Delta t$:

$$\psi(t + \Delta t) \approx \left(1 - \frac{i\hat{H}\Delta t}{\hbar}\right)\psi(t)$$

But the exact evolution operator $e^{-i\hat{H}\Delta t/\hbar}$ is **unitary** (preserves probability). Is our approximation?

Test: for a unitary operator $\hat{U}$, we need $\hat{U}^\dagger \hat{U} = \hat{I}$.

$$\hat{U}_{\text{Euler}}^\dagger \hat{U}_{\text{Euler}} = \left(1 + \frac{i\hat{H}\Delta t}{\hbar}\right)\left(1 - \frac{i\hat{H}\Delta t}{\hbar}\right) = 1 + \frac{\hat{H}^2 \Delta t^2}{\hbar^2} \neq \hat{I}$$

The norm *grows* at each step. Probability leaks in from nowhere — a numerical ghost with no physical meaning.

**Lesson:** The physics (unitarity) constrains which numerical methods are acceptable. This is a concrete example of the "translation layer" — the step from math to computation is not automatic; it requires physical reasoning.

> **For Lesson 1**, we sidestep this entirely by solving the **time-independent** equation, which is a matrix eigenvalue problem — no time stepping needed. But we flag this issue now because it will matter when we do time-dependent simulations later.

### A.5 The Time-Independent Schrödinger Equation

If we look for states with a *definite energy* — states whose probability distribution does not change in time — we can separate the time and space parts:

$$\psi(x,t) = \phi(x)\, e^{-iEt/\hbar}$$

The spatial part $\phi(x)$ satisfies:

$$\hat{H}\,\phi(x) = E\,\phi(x)$$

This is an **eigenvalue equation**: the Hamiltonian acting on an eigenstate returns the *same state* scaled by a number (the energy $E$). Not every state satisfies this — only special ones, the **eigenstates**. But any state can be written as a superposition of eigenstates, so solving this equation gives us the complete toolkit for the system.

For a particle in one dimension:

$$-\frac{\hbar^2}{2m}\frac{d^2\phi}{d x^2} + V(x)\,\phi(x) = E\,\phi(x)$$

**This is what we will solve numerically in this lesson.**

---
## Part B: From Continuous Equation to Computable Numbers

### B.1 The Problem: Particle in a 1D Infinite Square Well

The simplest quantum system: a particle of mass $m$ confined to a box of length $L$.

$$V(x) = \begin{cases} 0 & 0 < x < L \\ \infty & \text{otherwise} \end{cases}$$

The infinite walls mean $\phi(0) = \phi(L) = 0$ — the wavefunction must vanish at the boundaries.

**Analytical solution** (for later validation):

$$\phi_n(x) = \sqrt{\frac{2}{L}} \sin\left(\frac{n\pi x}{L}\right), \qquad E_n = \frac{n^2 \pi^2 \hbar^2}{2mL^2}, \qquad n = 1, 2, 3, \ldots$$

### B.2 Dimensional Analysis — Know the Answer Before You Compute

Before writing any code, we should know what *scale* of answer to expect.

The problem has three physical parameters: $\hbar$, $m$, and $L$. From dimensional analysis, the only energy scale we can construct is:

$$E_0 = \frac{\hbar^2}{2mL^2}$$

So all energies must be multiples of this. The ground state ($n=1$) has $E_1 = \pi^2 E_0$.

**Let's put in real numbers.** For an electron ($m = 9.109 \times 10^{-31}$ kg) in a box of $L = 1$ nm (roughly atomic scale):

$$E_0 = \frac{(1.055 \times 10^{-34})^2}{2 \times 9.109 \times 10^{-31} \times (10^{-9})^2} \approx 6.0 \times 10^{-29} \text{ J} \approx 0.038 \text{ eV}$$

So the ground state energy $E_1 = \pi^2 E_0 \approx 0.38$ eV. This is the right order of magnitude for quantum confinement effects in nanoscale systems — a good sanity check.

> **Habit:** Always estimate the answer from dimensional analysis before computing. If your code gives $10^{15}$ eV for an electron in a nanometer box, you have a bug.

### B.3 Natural Units — Making the Computer's Job Easier

Working in SI units means juggling numbers like $10^{-34}$ and $10^{-31}$. This invites floating-point trouble and makes code hard to read.

**Better approach:** Define natural units where the characteristic scales of the problem are $O(1)$.

For our problem, a convenient choice is **atomic units**:
- Length: bohr radius $a_0 = 0.5292$ Å $= 5.292 \times 10^{-11}$ m
- Energy: hartree $E_h = 27.21$ eV $= 4.360 \times 10^{-18}$ J
- Mass: electron mass $m_e = 1$ (by definition)
- $\hbar = 1$ (by definition)

In atomic units, the Schrödinger equation for an electron simplifies to:

$$-\frac{1}{2}\frac{d^2\phi}{dx^2} + V(x)\,\phi(x) = E\,\phi(x)$$

where $x$ is in bohr, $E$ is in hartree, and $V$ is in hartree. No factors of $10^{-34}$ anywhere.

A box of $L = 1$ nm $= 18.90$ bohr. The ground state energy:

$$E_1 = \frac{\pi^2}{2 \times 18.90^2} \approx 0.01384 \text{ hartree} = 0.377 \text{ eV}$$

Matches our SI estimate. Good.

> **Design principle:** Choose units that keep numbers $O(1)$. This makes code cleaner, debugging easier, and results more intuitive. Atomic units are standard in quantum chemistry and are what PySCF uses internally.

### B.4 Discretization — From Calculus to Linear Algebra

The Schrödinger equation is a differential equation on a continuous domain $x \in [0, L]$. A computer works with discrete arrays of numbers. We need to translate.

**Step 1: Define a grid.**

Choose $N$ equally spaced interior points: $x_j = j\,\Delta x$ for $j = 1, 2, \ldots, N$, where $\Delta x = L/(N+1)$.

The wavefunction becomes a vector: $\vec{\phi} = (\phi_1, \phi_2, \ldots, \phi_N)$ where $\phi_j \approx \phi(x_j)$.

Note: we don't include the boundary points $x_0 = 0$ and $x_{N+1} = L$ because we know $\phi = 0$ there.

**Step 2: Approximate the second derivative.**

The finite-difference approximation:

$$\frac{d^2\phi}{dx^2}\bigg|_{x_j} \approx \frac{\phi_{j-1} - 2\phi_j + \phi_{j+1}}{\Delta x^2}$$

This is a *local* approximation using three neighboring points. Its error is $O(\Delta x^2)$ — it gets better as the grid gets finer.

**Step 3: Assemble the Hamiltonian matrix.**

The kinetic energy operator $-\frac{1}{2}\frac{d^2}{dx^2}$ becomes the matrix:

$$T_{jk} = -\frac{1}{2\,\Delta x^2} \begin{pmatrix} -2 & 1 & 0 & \cdots \\ 1 & -2 & 1 & \cdots \\ 0 & 1 & -2 & \cdots \\ \vdots & & & \ddots \end{pmatrix} = \frac{1}{2\,\Delta x^2}\begin{pmatrix} 2 & -1 & 0 & \cdots \\ -1 & 2 & -1 & \cdots \\ 0 & -1 & 2 & \cdots \\ \vdots & & & \ddots \end{pmatrix}$$

The potential energy is diagonal: $V_{jk} = V(x_j)\,\delta_{jk}$.

The full Hamiltonian matrix is $\mathbf{H} = \mathbf{T} + \mathbf{V}$.

**Step 4: Solve the eigenvalue problem.**

$$\mathbf{H}\,\vec{\phi}_n = E_n\,\vec{\phi}_n$$

This is a standard matrix eigenvalue problem that NumPy/SciPy can solve in one function call.

> **What did we gain and what did we lose?** We gained computability — any potential $V(x)$ can be handled by just changing the diagonal of $\mathbf{V}$. We lost exactness — the finite grid spacing introduces error. But this error is controllable and quantifiable, which we will verify in Part D.

### B.5 Grid Parameters and Physical Accuracy

How fine must the grid be? The answer comes from physics:

The highest-energy eigenstate we can *resolve* on a grid with spacing $\Delta x$ has a wavelength $\lambda_{\min} = 2\,\Delta x$ (the Nyquist limit — same as in signal processing). The corresponding maximum energy is:

$$E_{\max} \sim \frac{\pi^2}{2\,(2\,\Delta x)^2} = \frac{\pi^2}{8\,\Delta x^2}$$

States with energy above this are aliased — their structure is finer than the grid can represent. So:

- **More grid points** → smaller $\Delta x$ → higher-energy states resolved accurately
- **Fewer grid points** → cheaper computation, but low-energy states only
- **For the lowest few eigenstates**, $N = 100$–$1000$ is typically more than enough

We'll verify this convergence behavior numerically in Part D.

---
## Part C: Building the Solver

### Design Principles

1. **Every line maps to physics or math from Parts A–B** — no magic
2. **Clean interface** — easy to change the potential, grid, or mass
3. **Returns results with units** — not just raw numbers
4. **Lightweight class** — holds the system definition, grid, and solutions together

### C.1 The Solver Code

We build a `QuantumSystem1D` class that encapsulates the physics.

In [ ]:
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt

class QuantumSystem1D:
    """
    Solve the 1D time-independent Schrödinger equation:
    
        -1/2 * d²φ/dx² + V(x) * φ(x) = E * φ(x)
    
    using finite-difference discretization and matrix diagonalization.
    
    Units: atomic units (ħ = mₑ = e = 1)
        - Length in bohr (1 bohr = 0.5292 Å)
        - Energy in hartree (1 hartree = 27.21 eV)
    
    Parameters
    ----------
    x_min, x_max : float
        Domain boundaries in bohr.
    N : int
        Number of interior grid points.
    V_func : callable
        Potential energy function V(x), takes array, returns array (in hartree).
    mass : float
        Particle mass in units of electron mass. Default = 1.0 (electron).
    """
    
    def __init__(self, x_min, x_max, N, V_func, mass=1.0):
        self.x_min = x_min
        self.x_max = x_max
        self.N = N
        self.mass = mass
        self.V_func = V_func
        
        # Build the grid (interior points only — boundary values are zero)
        self.dx = (x_max - x_min) / (N + 1)
        self.x = np.linspace(x_min + self.dx, x_max - self.dx, N)
        
        # Build and store the Hamiltonian
        self.H = self._build_hamiltonian()
        
        # Placeholder for solutions
        self.energies = None
        self.states = None
    
    def _build_hamiltonian(self):
        """
        Construct the Hamiltonian matrix H = T + V.
        
        Kinetic energy: finite-difference second derivative
            T_jk = (1 / 2m·dx²) × tridiagonal(-1, 2, -1)
        
        Potential energy: diagonal matrix
            V_jk = V(x_j) · δ_jk
        """
        # Kinetic energy matrix (tridiagonal)
        diag_main = np.full(self.N, 2.0)
        diag_off = np.full(self.N - 1, -1.0)
        T = np.diag(diag_main) + np.diag(diag_off, 1) + np.diag(diag_off, -1)
        T *= 1.0 / (2.0 * self.mass * self.dx**2)
        
        # Potential energy matrix (diagonal)
        V = np.diag(self.V_func(self.x))
        
        return T + V
    
    def solve(self, n_states=10):
        """
        Solve the eigenvalue problem H·φ = E·φ.
        
        Parameters
        ----------
        n_states : int
            Number of lowest eigenstates to return.
        
        Returns
        -------
        energies : ndarray, shape (n_states,)
            Eigenvalues in hartree.
        states : ndarray, shape (N, n_states)
            Eigenvectors (columns), normalized on the grid.
        """
        # Use eigh (for Hermitian/symmetric matrices) — guarantees real eigenvalues
        # and orthogonal eigenvectors, which is what physics demands.
        eigenvalues, eigenvectors = linalg.eigh(self.H)
        
        # Keep only the lowest n_states
        self.energies = eigenvalues[:n_states]
        self.states = eigenvectors[:, :n_states]
        
        # Normalize: ∫|φ|² dx ≈ Σ|φⱼ|² · dx = 1
        for i in range(n_states):
            norm = np.sqrt(np.sum(self.states[:, i]**2) * self.dx)
            self.states[:, i] /= norm
        
        return self.energies, self.states
    
    def plot_states(self, n_show=4, offset_by_energy=True):
        """
        Plot wavefunctions and probability densities.
        
        Parameters
        ----------
        n_show : int
            Number of states to plot.
        offset_by_energy : bool
            If True, vertically offset each wavefunction by its energy.
        """
        if self.energies is None:
            self.solve(n_states=n_show)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharey=offset_by_energy)
        
        # Include boundary points (zero) for clean plots
        x_full = np.concatenate([[self.x_min], self.x, [self.x_max]])
        
        for i in range(min(n_show, len(self.energies))):
            E = self.energies[i]
            phi = self.states[:, i]
            phi_full = np.concatenate([[0], phi, [0]])
            
            offset = E if offset_by_energy else 0
            
            # Wavefunction
            ax1.plot(x_full, phi_full + offset, label=f'n={i+1}, E={E:.4f} Ha')
            ax1.axhline(y=offset, color='gray', linewidth=0.3)
            
            # Probability density
            ax2.plot(x_full, phi_full**2 + offset, label=f'n={i+1}')
            ax2.axhline(y=offset, color='gray', linewidth=0.3)
        
        ax1.set_xlabel('x (bohr)')
        ax1.set_ylabel('φ(x) + E (hartree)' if offset_by_energy else 'φ(x)')
        ax1.set_title('Wavefunctions')
        ax1.legend(fontsize=8)
        
        ax2.set_xlabel('x (bohr)')
        ax2.set_ylabel('|φ(x)|² + E' if offset_by_energy else '|φ(x)|²')
        ax2.set_title('Probability Densities')
        ax2.legend(fontsize=8)
        
        plt.tight_layout()
        plt.savefig('figures/eigenstates.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        return fig

### C.2 Tracing the Code Back to the Physics

Let's make the mapping explicit:

| Physics / Math | Code |
|---------------|------|
| Grid $x_j = x_{\min} + j\,\Delta x$ | `self.x = np.linspace(...)` |
| Kinetic energy $\frac{1}{2m\,\Delta x^2}\text{tridiag}(-1,2,-1)$ | `T = np.diag(...) + np.diag(...)` scaled by `1/(2·mass·dx²)` |
| Potential energy $V(x_j)\delta_{jk}$ | `V = np.diag(self.V_func(self.x))` |
| Hamiltonian $\mathbf{H} = \mathbf{T} + \mathbf{V}$ | `return T + V` |
| Eigenvalue problem $\mathbf{H}\vec{\phi} = E\vec{\phi}$ | `linalg.eigh(self.H)` |
| Normalization $\int|\phi|^2 dx = 1$ | `norm = np.sqrt(np.sum(...) * self.dx)` |

We use `eigh` (not `eig`) because the Hamiltonian is Hermitian (symmetric for real potentials). This is a physics constraint — the Hamiltonian must be Hermitian to guarantee real energy eigenvalues and orthogonal eigenstates. Using `eigh` tells the computer to exploit this structure.

### C.3 Running the Solver: Electron in a 1 nm Box

In [ ]:
# === Physical setup ===
# 1 nm box in atomic units
L_nm = 1.0                          # box length in nanometers
bohr_per_nm = 18.8973               # conversion factor
L = L_nm * bohr_per_nm              # box length in bohr

# Define the potential: zero inside the box
# (boundaries are handled by the grid — φ=0 at endpoints)
def V_box(x):
    return np.zeros_like(x)

# === Create and solve the system ===
system = QuantumSystem1D(x_min=0, x_max=L, N=200, V_func=V_box)
energies, states = system.solve(n_states=6)

# === Display results ===
ha_to_eV = 27.2114                  # conversion factor

print("Particle in a 1D box: L = {:.2f} bohr = {:.2f} nm".format(L, L_nm))
print("=" * 55)
print(f"{'n':>3}  {'E (hartree)':>14}  {'E (eV)':>10}  {'E/E₀':>8}")
print("-" * 55)

E0 = np.pi**2 / (2 * L**2)         # energy unit ℏ²π²/(2mL²)
for i, E in enumerate(energies):
    n = i + 1
    print(f"{n:>3}  {E:>14.6f}  {E * ha_to_eV:>10.4f}  {E/E0:>8.3f}")

print("-" * 55)
print(f"E₀ = π²ħ²/(2mL²) = {E0:.6f} hartree = {E0*ha_to_eV:.4f} eV")
print(f"Expected ratio E_n/E₀ = n² : {', '.join(str(n**2) for n in range(1,7))}")

In [ ]:
# === Plot eigenstates ===
system.plot_states(n_show=4)

---
## Part D: Validation — Trust but Verify

### D.1 Compare to Analytical Solution

For the infinite square well, we know the exact answer: $E_n = n^2 \pi^2 / (2L^2)$ in atomic units. Let's quantify the numerical error.

In [ ]:
# === Analytical vs numerical comparison ===
print("Validation: Numerical vs Analytical Energies")
print("=" * 65)
print(f"{'n':>3}  {'E_numerical':>14}  {'E_analytical':>14}  {'Rel. Error':>12}")
print("-" * 65)

for i, E_num in enumerate(energies):
    n = i + 1
    E_exact = n**2 * np.pi**2 / (2 * L**2)
    rel_err = abs(E_num - E_exact) / E_exact
    print(f"{n:>3}  {E_num:>14.8f}  {E_exact:>14.8f}  {rel_err:>12.2e}")

### D.2 Convergence Study — How Does Grid Refinement Affect Accuracy?

The discretization error should decrease as we use more grid points. Let's verify this and measure the convergence rate.

In [ ]:
# === Convergence study ===
N_values = [20, 50, 100, 200, 500, 1000]
errors_n1 = []  # error in ground state
errors_n3 = []  # error in 3rd state

E1_exact = np.pi**2 / (2 * L**2)
E3_exact = 9 * np.pi**2 / (2 * L**2)

for N in N_values:
    sys_test = QuantumSystem1D(0, L, N, V_box)
    E_test, _ = sys_test.solve(n_states=4)
    errors_n1.append(abs(E_test[0] - E1_exact) / E1_exact)
    errors_n3.append(abs(E_test[2] - E3_exact) / E3_exact)

# Plot convergence
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(N_values, errors_n1, 'o-', label='n=1 (ground state)')
ax.loglog(N_values, errors_n3, 's-', label='n=3')

# Reference line: O(N⁻²) convergence (since error ~ Δx² ~ (L/N)²)
N_ref = np.array(N_values, dtype=float)
ax.loglog(N_ref, 0.5 * (N_ref[0]/N_ref)**2 * errors_n1[0], 
          '--', color='gray', label='$O(N^{-2})$ reference')

ax.set_xlabel('Number of grid points N')
ax.set_ylabel('Relative error in energy')
ax.set_title('Convergence of Finite-Difference Eigenvalues')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nWith N=200: ground state error = {errors_n1[3]:.2e}")
print(f"With N=200: n=3 state error    = {errors_n3[3]:.2e}")

### D.3 Sanity Checks — Building Good Habits

Beyond comparing to analytical results, we should always check:

1. **Orthonormality**: eigenstates should satisfy $\langle \phi_m | \phi_n \rangle = \delta_{mn}$
2. **Node counting**: the $n$-th eigenstate should have $n-1$ nodes (zero crossings)
3. **Symmetry**: for a symmetric potential, eigenstates alternate even/odd

In [ ]:
# === Orthonormality check ===
print("Orthonormality check: <φ_m | φ_n> (should be δ_mn)")
print("=" * 50)
n_check = 4
overlap = np.zeros((n_check, n_check))
for m in range(n_check):
    for n in range(n_check):
        overlap[m, n] = np.sum(states[:, m] * states[:, n]) * system.dx

# Print as a formatted matrix
for m in range(n_check):
    row = "  ".join(f"{overlap[m,n]:>8.5f}" for n in range(n_check))
    print(f"  m={m+1}: {row}")

# === Node counting ===
print(f"\nNode count check (n-th state should have n-1 nodes):")
for i in range(4):
    phi = states[:, i]
    # Count sign changes
    nodes = np.sum(np.diff(np.sign(phi)) != 0)
    print(f"  n={i+1}: {nodes} nodes (expected {i})")

---
## Part E: What If? — Beyond Analytical Solutions

The real power of a numerical solver is handling potentials that have *no* analytical solution. With our tool, changing the potential is trivial — just write a new `V_func`.

### E.1 Asymmetric Potential Well

What happens if the box isn't flat inside?

In [ ]:
# === Asymmetric potential: a sloped floor ===
def V_slope(x):
    """Linear potential inside the box — like gravity."""
    V_depth = 0.01  # slope in hartree/bohr — gentle tilt
    return V_depth * (x - L/2)

system_slope = QuantumSystem1D(0, L, 200, V_slope)
energies_slope, states_slope = system_slope.solve(n_states=6)

# Compare to flat box
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot the potential
x_full = np.concatenate([[0], system_slope.x, [L]])

for ax, sys, title, engs in [
    (axes[0], system, "Flat box", energies),
    (axes[1], system_slope, "Sloped floor", energies_slope)
]:
    # Plot potential
    V_vals = np.concatenate([[0], sys.V_func(sys.x), [0]])
    ax.fill_between(x_full, V_vals - 0.002, V_vals, alpha=0.2, color='gray', label='V(x)')
    
    for i in range(4):
        E = engs[i]
        phi = sys.states[:, i]
        phi_full = np.concatenate([[0], phi, [0]])
        scale = 0.003  # scale wavefunctions for visibility
        ax.plot(x_full, phi_full * scale + E, label=f'n={i+1}, E={E:.5f}')
        ax.axhline(y=E, color='gray', linewidth=0.3, linestyle='--')
    
    ax.set_xlabel('x (bohr)')
    ax.set_ylabel('Energy (hartree)')
    ax.set_title(title)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('figures/comparison_flat_vs_slope.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEnergy comparison (hartree):")
print(f"{'n':>3}  {'Flat box':>12}  {'Sloped':>12}  {'Shift':>12}")
print("-" * 45)
for i in range(4):
    print(f"{i+1:>3}  {energies[i]:>12.6f}  {energies_slope[i]:>12.6f}  {energies_slope[i]-energies[i]:>12.6f}")

### E.2 Double Well — Tunneling

One of the most important quantum phenomena: a particle can "tunnel" through a barrier that classical mechanics says is impenetrable.

In [ ]:
# === Double well potential ===
def V_double_well(x):
    """Two wells separated by a barrier."""
    center = L / 2
    barrier_height = 0.05  # hartree
    barrier_width = L / 10  # bohr
    V = np.zeros_like(x)
    V[np.abs(x - center) < barrier_width / 2] = barrier_height
    return V

system_dw = QuantumSystem1D(0, L, 500, V_double_well)
energies_dw, states_dw = system_dw.solve(n_states=6)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
x_full = np.concatenate([[0], system_dw.x, [L]])
V_full = np.concatenate([[0.1], V_double_well(system_dw.x), [0.1]])

ax.fill_between(x_full, -0.001, V_full, alpha=0.15, color='orange', label='V(x)')
ax.plot(x_full, V_full, 'k-', linewidth=0.5)

scale = 0.002
for i in range(4):
    E = energies_dw[i]
    phi = states_dw[:, i]
    phi_full = np.concatenate([[0], phi, [0]])
    ax.plot(x_full, phi_full * scale + E, label=f'n={i+1}, E={E:.6f} Ha')
    ax.axhline(y=E, color='gray', linewidth=0.3, linestyle='--')

ax.set_xlabel('x (bohr)')
ax.set_ylabel('Energy (hartree)')
ax.set_title('Double Well: Tunneling Splits the Degeneracy')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('figures/double_well.png', dpi=150, bbox_inches='tight')
plt.show()

# The key observable: tunnel splitting
print(f"\nTunnel splitting: ΔE = E₂ - E₁ = {energies_dw[1] - energies_dw[0]:.6f} hartree "
      f"= {(energies_dw[1] - energies_dw[0])*ha_to_eV*1000:.2f} meV")
print("Notice: the first two states are nearly degenerate (symmetric/antisymmetric pair)")
print("The splitting decreases exponentially with barrier height and width — this is tunneling.")

---
## Summary

### What We Built

A clean, general-purpose 1D quantum solver in ~60 lines of core code. The same code handles:
- Infinite square well (analytical solution exists — for validation)
- Asymmetric potentials (no analytical solution)
- Double wells and tunneling (real quantum phenomena)

### The Translation Layer — Key Takeaways

| Principle | What we did |
|-----------|------------|
| **Units matter** | Work in atomic units to keep numbers $O(1)$ |
| **Estimate before computing** | Dimensional analysis gives expected energy scale |
| **Discretization = approximation** | Continuous $\to$ discrete introduces controlled error |
| **Physics constrains numerics** | Hermitian matrix → use `eigh`; unitarity → careful time steppers |
| **Validate ruthlessly** | Compare to analytical results, check orthonormality, count nodes |

### What Comes Next

- **Lesson 2:** Harmonic oscillator — new potential, same solver. Introduction to perturbation theory (numerical vs analytical).
- **Lesson 3:** Multi-dimensional systems. Hydrogen atom. Separation of variables vs brute-force grids.
- **Future:** Many-electron systems, basis sets, Hartree-Fock, and eventually PySCF.

The computational pattern — *build Hamiltonian → diagonalize → analyze* — scales all the way up. What changes is how we construct the Hamiltonian and what basis we use to represent it.

---
## Exercises

1. **Change the box size.** How do the energies scale with $L$? Verify the $1/L^2$ dependence numerically.

2. **Heavier particle.** Change `mass` to 1836 (proton mass in atomic units). How do the energies change? Why?

3. **Finite well.** Replace the infinite well with a finite one: $V(x) = 0$ for $|x - L/2| < w/2$, $V(x) = V_0$ otherwise (make the domain larger than the well). How many bound states exist for a given $V_0$ and $w$?

4. **Harmonic oscillator.** Use $V(x) = \frac{1}{2}\omega^2 x^2$ (set the domain large enough that the wavefunction dies off before the boundary). Compare to the exact result $E_n = \omega(n + 1/2)$.

5. **Convergence rate.** The finite-difference error scales as $O(\Delta x^2)$. Can you implement a 4th-order scheme and show $O(\Delta x^4)$ convergence?